<a href="https://colab.research.google.com/github/mahmoadalmasry2020-cyber/Fast-tokenizers-special-powers/blob/main/Fast_tokenizers%E2%80%99_special_powers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"

tokenizer=AutoTokenizer.from_pretrained(model_checkpoint)
model=AutoModelForTokenClassification.from_pretrained(model_checkpoint)
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."

inputs=tokenizer(example,return_tensors="pt")
outputs=model(**inputs)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
import torch

probabilities=torch.nn.functional.softmax(outputs.logits, dim=-1)[0].tolist()
predictions=outputs.logits.argmax(dim=-1)[0].tolist()
print(predictions)

[0, 0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 6, 6, 6, 0, 8, 0, 0]


In [13]:
import numpy as np

results=[]
inputs_with_offsets=tokenizer(example,return_offsets_mapping=True)
tokens=inputs_with_offsets.tokens()
offsets=inputs_with_offsets["offset_mapping"]

idx=0
while idx<len(predictions):
    pred=predictions[idx]
    label=model.config.id2label[pred]
    if label != "O":
       label=label[2:]
       start,_=offsets[idx]

       all_scores=[]

       while(
          idx < len(predictions)
          and model.config.id2label[predictions[idx]] == f"I-{label}"
      ):
           all_scores.append(probabilities[idx][pred])
           _,end=offsets[idx]
           idx+=1
       score=np.mean(all_scores).item()
       word=example[start:end]

       results.append(
          {
              "entity_group":label,
              "score":score,
              "word":word,
              "start":start,
              "end":end,


          }
       )
    idx+=1
print(results)


[{'entity_group': 'PER', 'score': 0.998169407248497, 'word': 'Sylvain', 'start': 11, 'end': 18}, {'entity_group': 'ORG', 'score': 0.9796019991238912, 'word': 'Hugging Face', 'start': 33, 'end': 45}, {'entity_group': 'LOC', 'score': 0.9932106137275696, 'word': 'Brooklyn', 'start': 49, 'end': 57}]
